# 05 · Choose lessons from unlabeled GAVD video

**Goal:** deploy the frozen selectors, adapt each student with its
selected lessons, and save predictions before opening reference labels.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](README.md)
· [HAIC setup and launch commands](../../slurm/synthetic-training/README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

## 1. Use only information available at deployment

Set `ST_STUDENT_ID` to a configured estimator. The deployment array can
include the held architecture. Each collection supplies its context
clips. Measure the common probe response on those clips and use the
frozen source teacher to choose a lesson at the declared budget.

Labeled synthetic diagnostics remain allowed deployment inputs. They
were fixed before real evaluation and do not contain GAVD references.
We are testing a supervised adaptation procedure chosen using unlabeled
target video, not label-free student training.

In [ ]:
STUDENT_ID = os.environ.get("ST_STUDENT_ID")
if not STUDENT_ID:
    raise ValueError("Set ST_STUDENT_ID to a configured deployment student.")
display(pd.DataFrame([cfg.student(STUDENT_ID)]))

## 2. Run equal-budget branches and save real predictions

Every mechanism comparison adapts the same post-probe checkpoint for
the same remaining number of updates. Replay-only, the original
estimator, probe-only, and full-budget replay preserve the practical
comparison. Fixed/random/balanced choices test ordinary augmentation.

Select once for a collection, then evaluate on its separate recordings.
Do not choose a different lesson after seeing a reference frame's error.
When selectors choose the same lesson, their shared prediction result
is expected; it is not evidence that one information source helped.

In [ ]:
started = perf_counter()
deployment = workflow.deploy(cfg, student_id=STUDENT_ID)
show_result(deployment)
print(f"Deployment took {(perf_counter() - started) / 3600:.2f} hours.")

## 3. Keep decision and measurement separate

The saved choices and predictions are the outputs of deployment.
Notebook 06 reads independent human references. This stage must not
search all real lesson outcomes to decide which lesson would have won.

The primary teacher and comparator were chosen on source validation.
If an early real result prompts a method change, declare the early set
as labeled development and retain confirmation recordings untouched.

Once all students have completed deployment, continue to
[08 · Exchange selected lessons](08_exchange_selected_lessons.ipynb).
Then explicitly select `early` or `confirmation` in notebook 06.